# Hybrid L2/3 circuit on a Colab GPU

This notebook runs the isolated Allen V1 layer 2/3 circuit with **ANN pyramidal cells** (the Standard
CNN-LSTM → Izhikevich + conductance-branch model, all 56,057 PCs advanced together every 1 ms) and
**GLIF interneurons** (2,927 Pvalb, 2,120 Sst, 4,845 Htr3a, run in NEST at 0.25 ms). It then makes the
figures and statistics of the run: rates per class, the PC firing-rate distribution (silent fraction,
top-10% spike share, 99th percentile, max) against the GLIF network and the Allen Neuropixels L2/3 RS
units, per-cell agreement with the GLIF network, and where the run time went.

**Before you start**
1. Upload the bundle `hybrid_l23_bundle.zip` to Google Drive, into `MyDrive/hybrid_l23_colab/`.
2. *Runtime → Change runtime type → GPU* (T4 is fine, L4/A100 are faster). High RAM is not needed: a run
   uses about 6–8 GB.
3. Run the cells from top to bottom. Pick the condition in **Step 4**.

**What runs where.** The PCs (≈93% of the work) run on the GPU through TensorFlow. The interneurons stay
in NEST on the CPU, so a 3 s run cannot go below ≈1.5–2 min (NEST ≈ 90 s, spike routing ≈ 10 s) however
fast the GPU is. On the 4-core CPU machine the same run took 43 min (homogeneous) to 60+ min
(heterogeneous library).

**The model is unchanged.** This is the same code as `Isolated L2_3 circuit/circuit/hybrid.py` in the
repository (copied into the bundle), with the same weights. On a GPU the floating-point operations run in a
different order, so individual spikes can come out at different times than in a CPU run (the network
amplifies tiny differences), but the statistics should match. Step 5 checks this on a few cells.

Note: since 2026-11-26 the default single-cell architecture also routes synaptic input through the
hardcoded synaptic kernel layer (PR 22). The circuit code in this bundle still feeds the Allen alpha-PSC
currents into the CNN-LSTM (the version used for the Aim 2 runs). When the kernel-layer circuit is merged,
rebuild the bundle with `make_bundle.py` and this notebook runs that version.

## Step 1. Check the GPU
If this prints no GPU, change the runtime type (see above) and run it again.

In [ ]:
import subprocess, shutil
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi not found: this runtime has no GPU (the notebook still runs, at CPU speed)")
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
# Let TensorFlow take GPU memory as it needs it instead of all at once.
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)

## Step 2. Install NEST and BMTK
TensorFlow, numpy, h5py, pandas, scipy and matplotlib are already in Colab. NEST 3.10 and BMTK 1.2 come from
PyPI (about 1 minute). The circuit was built and checked with these versions.

In [ ]:
!pip -q install "nest-simulator==3.10.*" "bmtk==1.2.*"
import nest, bmtk
print("NEST", nest.__version__, "| BMTK", bmtk.__version__)

## Step 3. Copy the bundle from Drive to the Colab disk
The bundle holds everything the run needs:

| folder | contents |
|---|---|
| `code/` | the repository's `Isolated L2_3 circuit` code: `circuit/` (hybrid circuit), `Standard/` (the Standard model and its weights), `experiments/population/library/` (the 41 fitted library cells), `experiments/aim2_controls/` (population files that assign a fitted model to each PC) |
| `data/prep/` | the prepared hybrid circuit: NEST network of the interneurons (`nest_network/`, `config_nest_part.json`), PC → PC and interneuron → PC edges (`pc_to_pc.npz`, `in_to_pc.npz`), and the precomputed external input current onto every PC (`pc_external_current.npy`, 3000 ms × 56,057 PCs) |
| `data/allen/` | the Allen files NEST needs: GLIF and synapse parameters (`components/`), the LGN grating spikes and the background spikes (trial 0) |
| `data/streams/` | the replayed L1/L4/L5/L6 spikes (`v1ext_spikes.h5`) and the GLIF network's own L2/3 spikes (`l23_reference_spikes.h5`, the reference) |
| `data/cells/` | class and depth of every L2/3 cell (`l23_cells.npz`), and the Neuropixels L2/3 RS rates (`npx_l23_rs_rates_all.npy`, 595 units) |

Working from the local disk is much faster than reading the files over Drive. Unzipping takes 1–2 minutes.

In [ ]:
import os, json, time, shutil, glob, sys
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/hybrid_l23_colab"   # where you put the bundle; results go here too
BUNDLE_ZIP = os.path.join(DRIVE_DIR, "hybrid_l23_bundle.zip")
WORK = "/content/hybrid_l23"                               # local copy

if not os.path.exists(os.path.join(WORK, "BUNDLE_OK")):
    assert os.path.exists(BUNDLE_ZIP), f"upload hybrid_l23_bundle.zip to {DRIVE_DIR} first"
    t = time.time()
    shutil.copy(BUNDLE_ZIP, "/content/bundle.zip")
    !unzip -q -o /content/bundle.zip -d /content
    os.remove("/content/bundle.zip")
    open(os.path.join(WORK, "BUNDLE_OK"), "w").close()
    print(f"bundle unpacked in {time.time() - t:.0f} s")

CODE, DATA = os.path.join(WORK, "code"), os.path.join(WORK, "data")
PREP = os.path.join(DATA, "prep")

# The NEST config was written with the placeholder @BUNDLE@ for the bundle folder; point it at WORK.
src = os.path.join(PREP, "config_nest_part.template.json")
with open(src) as f:
    cfg_text = f.read().replace("@BUNDLE@", WORK)
with open(os.path.join(PREP, "config_nest_part.json"), "w") as f:
    f.write(cfg_text)
missing = [v for d in json.loads(cfg_text)["networks"]["nodes"] for v in d.values() if not os.path.exists(v)]
assert not missing, missing
print(json.load(open(os.path.join(PREP, "prepare.json"))))

## Step 4. Choose what to run

Each condition is a set of PC models plus the synaptic gain that matched the GLIF network's mean PC rate
(tuned on 3 s runs; see the repository README of `circuit/` and `experiments/aim2_controls/`). Every
condition keeps the PC → interneuron synapses at Allen's weights (`PC_TO_IN_GAIN = 1.0`).

| condition | PCs | gain onto PCs |
|---|---|---|
| `homogeneous` | every PC is the Standard model (Allen cell 487661754) | 0.848 |
| `library_shuffled` | 38 library cells (population fits of real Allen recordings), 1/38 of the PCs each, placed at random | 0.8103 |
| `library_depth` | the same 38 cells, each PC gets the model whose soma depth is closest | 0.8103 |
| `library_rheo_matched` | as `library_shuffled`, but each PC's input is scaled by its model's rheobase / 90 pA (removes the threshold spread) | 0.8813 |
| `eight_fits_shuffled` | the earlier 8-fit set (Standard + 5 real-recording fits + 2 simulated all-active fits) | 1.284 |

`TSTOP_MS` is the simulated time. The stimulus is 0.5 s gray screen, then the 315° grating from 0.5 s;
the prepared input covers 3000 ms. For a quick test use 300 ms.

In [ ]:
CONDITION = "homogeneous"
TSTOP_MS = 3000
PC_TO_IN_GAIN = 1.0

AIM2 = os.path.join(CODE, "experiments", "aim2_controls")
CONDITIONS = {
    "homogeneous":          dict(population=None, gain=0.848),
    "library_shuffled":     dict(population=os.path.join(AIM2, "populations_library", "shuffled.json"), gain=0.8103),
    "library_depth":        dict(population=os.path.join(AIM2, "populations_library", "depth.json"), gain=0.8103),
    "library_rheo_matched": dict(population=os.path.join(AIM2, "populations_library", "rheo_matched.json"), gain=0.8813),
    "eight_fits_shuffled":  dict(population=os.path.join(AIM2, "populations", "shuffled.json"), gain=1.284),
}
cond = CONDITIONS[CONDITION]
RUN_NAME = f"{CONDITION}_{TSTOP_MS}ms_{time.strftime('%Y%m%d-%H%M')}"
OUT = os.path.join(WORK, "runs", RUN_NAME)              # local, copied to Drive at the end
OUT_DRIVE = os.path.join(DRIVE_DIR, "runs", RUN_NAME)
os.makedirs(OUT, exist_ok=True)
print(CONDITION, cond, "->", OUT_DRIVE)

## Step 5. Check the PC layer on this GPU, and pick the faster front-end mode

Two short checks before the long run (about 1–2 minutes together):

1. **Same model.** `BatchedStandard` (one 1 ms step for many cells at a time, what the circuit uses) must give
   the same spikes as `StandardCell.simulate` (the whole trace at once, how the model was trained and
   validated). The repository test checks this on CPU, where it is exact. Here we report how many spikes agree
   within 1 ms on 6 cells × 400 ms of fluctuating current.
2. **Speed.** The CNN-LSTM front end can run XLA-compiled (`jit`, the default) or through TensorFlow's
   standard kernels, which on a GPU can use cuDNN's fused LSTM. We time one 1 ms step for all 56,057 PCs both
   ways and use the faster one. Both compute the same function.

In [ ]:
sys.path.insert(0, CODE); sys.path.insert(0, os.path.join(CODE, "Standard"))
import numpy as np
import standard_model as sm
from circuit.pc_layer import BatchedStandard

cell = sm.load(os.path.join(CODE, "Standard"))
rng = np.random.default_rng(0)
T, n = 400, 6
I = 60 + 120 * rng.standard_normal((n, T)).cumsum(1) / np.sqrt(np.arange(1, T + 1))
I = np.clip(I, -100, 400).astype(np.float32)
_, ref = cell.simulate(I)
b = BatchedStandard(cell, n, I_first=I[:, 0])
got = np.stack([b.step(I[:, t]) for t in range(T)], 1)
def matched(a, b):
    """Spikes of a that have a spike of b within 1 ms."""
    ta, tb = np.flatnonzero(a), np.flatnonzero(b)
    return int((np.abs(ta[:, None] - tb[None]).min(1) <= 1).sum()) if len(ta) and len(tb) else 0
ok = sum(matched(got[i], ref[i]) for i in range(n))
print(f"check 1: {ref.sum()} spikes in the whole-trace model, {got.sum()} in the per-step layer, "
      f"{ok} of them within 1 ms; identical: {np.array_equal(got, ref)}")

N_PC = 56057
I_bench = rng.normal(100, 50, (12, N_PC)).astype(np.float32)
bench = {}
for jit in (True, False):
    layer = BatchedStandard(cell, N_PC, jit=jit)
    for k in range(2):
        layer.step(I_bench[k])          # first calls compile
    t = time.time()
    for k in range(2, 12):
        layer.step(I_bench[k])
    bench[jit] = (time.time() - t) / 10
    del layer
FRONT_JIT = min(bench, key=bench.get)
print("check 2: one step for all PCs: " + ", ".join(f"{'jit' if k else 'no jit'} {v * 1e3:.0f} ms" for k, v in bench.items())
      + f" -> using {'jit' if FRONT_JIT else 'no jit'}")
print(f"expected PC time for {TSTOP_MS} ms: about {bench[FRONT_JIT] * TSTOP_MS / 60:.1f} min "
      f"(heterogeneous conditions run one batch per model, so they take longer), plus about "
      f"{0.032 * TSTOP_MS / 60:.1f} min for NEST and spike routing")

## Step 6. Run the circuit

Each simulated millisecond:
1. every PC's input current for (k, k+1] = the precomputed replayed input (L1/L4/L5/L6, LGN, background) +
   the PC and interneuron spikes arriving in that millisecond, through the fixed Allen alpha-PSC synaptic
   layer; everything onto PCs is multiplied by the gain;
2. one batched Standard step for all PCs (on the GPU) → PC spikes, stamped at k+1;
3. the PC spikes go to NEST as spike-generator times, NEST advances the interneurons by 1 ms (4 steps of
   0.25 ms), and their spikes are routed to the PCs' synaptic layer with each edge's delay.

The run is a separate process (so it can be rerun without restarting NEST in this notebook). Progress is
printed every 100 ms: mean PC and interneuron rates so far, mean PC input, seconds per simulated ms.
Keep the browser tab open: free Colab sessions stop when the tab is closed or idle for long.

In [ ]:
cmd = [sys.executable, "-m", "circuit.hybrid", "run", "--prep", PREP, "--standard-dir", os.path.join(CODE, "Standard"),
       "--gain", str(cond["gain"]), "--pc-to-in-gain", str(PC_TO_IN_GAIN), "--tstop", str(TSTOP_MS),
       "--out", OUT, "--threads", "1", "--log-every", "100",
       "--synapses", "current"]   # this package's gains were tuned on the current path (PR 9/14), not the kernel layer
if cond["population"]:
    cmd += ["--population", cond["population"]]
if not FRONT_JIT:
    cmd += ["--no-front-jit"]
env = dict(os.environ, PYTHONPATH=CODE, TF_CPP_MIN_LOG_LEVEL="2")
print(" ".join(cmd))
t_run = time.time()
proc = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(os.path.join(OUT, "log.txt"), "w") as log:
    for line in proc.stdout:
        log.write(line)
        if line.startswith("[run") or line.startswith("{") or "Error" in line or "error" in line:
            print(line, end="")
assert proc.wait() == 0, f"run failed, see {OUT}/log.txt"
print(f"done in {(time.time() - t_run) / 60:.1f} min")
os.makedirs(OUT_DRIVE, exist_ok=True)
for f in glob.glob(os.path.join(OUT, "*")):
    shutil.copy(f, OUT_DRIVE)
print("raw outputs copied to", OUT_DRIVE)

## Step 7. Load the spikes

`spikes.h5` (SONATA) holds every L2/3 spike of the run, with the cells' L2/3 node ids (0–65,948), the same ids
as the GLIF network's reference spikes. `l23_cells.npz` gives each id its class and soma depth. Rates in the
distribution statistics use the grating period 0.5–3 s (0.5–`TSTOP_MS` for shorter runs, the whole run if it is
shorter than 1 s), as in the earlier in vivo comparison and the Aim 2 controls. A cell with fewer than 0.4 Hz (one spike in 2.5 s) counts as silent.

In [ ]:
import h5py
import matplotlib.pyplot as plt
from circuit.hybrid import _read_spikes

cells = np.load(os.path.join(DATA, "cells", "l23_cells.npz"), allow_pickle=True)
cls = cells["cls"]                     # "L2/3 E", "L2/3 Pvalb", "L2/3 Sst", "L2/3 Htr3a" per L2/3 id
N_L23 = len(cls)
CLASSES = ["L2/3 E", "L2/3 Pvalb", "L2/3 Sst", "L2/3 Htr3a"]
COLORS = {"L2/3 E": "#c0392b", "L2/3 Pvalb": "#2471a3", "L2/3 Sst": "#229954", "L2/3 Htr3a": "#d68910"}
T_END = float(TSTOP_MS)
# grating window for the distribution statistics; a short test run (< 1 s) uses the whole run instead
T0, T1 = (500.0, T_END) if T_END >= 1000 else (0.0, T_END)
SILENT = 0.4

def load(path, t_end=T_END):
    ids, ts = _read_spikes(path, "l23")
    m = ts < t_end
    return ids[m], ts[m]

hyb = load(os.path.join(OUT, "spikes.h5"))
glif = load(os.path.join(DATA, "streams", "l23_reference_spikes.h5"))
invivo = np.load(os.path.join(DATA, "cells", "npx_l23_rs_rates_all.npy"))   # Hz, drifting gratings, 595 units
run_info = json.load(open(os.path.join(OUT, "run.json")))

def rates(sp, t0, t1):
    ids, ts = sp
    m = (ts >= t0) & (ts < t1)
    return np.bincount(ids[m], minlength=N_L23)[:N_L23] / ((t1 - t0) / 1000)

is_pc = cls == "L2/3 E"
print(f"{len(hyb[0])} spikes in the hybrid run, {len(glif[0])} in the GLIF network (0-{T_END:.0f} ms)")

## Step 8. Statistics

* **Rates per class** over the whole run, hybrid vs GLIF network.
* **PC rate distribution** during the grating (same definitions as `experiments/aim2_controls/analyze.py`):
  mean, median, silent fraction, 90th/99th percentile, max, share of all PC spikes fired by the top 10% and
  top 1% of PCs, and the SD and skew of log10 rate among active cells. The in vivo sample has 595 units, so
  the model statistics are also given at that sample size (2,000 random draws of 595 PCs, median and 95%
  interval): the max and the top-1% share depend strongly on sample size.
* **Per-PC agreement** with the GLIF network (Pearson r of rates, Spearman ρ).
* For heterogeneous conditions, the **mean rate of the PCs of each model**.

In [ ]:
from scipy.stats import spearmanr, pearsonr, ks_2samp

def dist_stats(r):
    r = np.asarray(r, float)
    a = np.sort(r)[::-1]
    k = max(1, int(round(0.1 * len(a))))
    lg = np.log10(r[r >= SILENT])
    return dict(n=int(len(r)), mean=float(r.mean()), median=float(np.median(r)), frac_silent=float((r < SILENT).mean()),
                p90=float(np.percentile(r, 90)), p99=float(np.percentile(r, 99)), max=float(r.max()),
                top10_share=float(a[:k].sum() / max(a.sum(), 1e-12)),
                top1_share=float(a[:max(1, len(a) // 100)].sum() / max(a.sum(), 1e-12)),
                sd_log10_active=float(lg.std()), skew_log10_active=float(((lg - lg.mean()) ** 3).mean() / lg.std() ** 3))

def at_n(r, n, reps=2000, seed=0):
    g = np.random.default_rng(seed)
    keys = ("mean", "frac_silent", "p99", "max", "top10_share")
    v = {k: [] for k in keys}
    for _ in range(reps):
        s = dist_stats(g.choice(r, n, replace=False))
        for k in keys:
            v[k].append(s[k])
    return {k: [float(np.percentile(x, 2.5)), float(np.median(x)), float(np.percentile(x, 97.5))] for k, x in v.items()}

r_all_h, r_all_g = rates(hyb, 0, T_END), rates(glif, 0, T_END)
per_class = {c: dict(hybrid_hz=float(r_all_h[cls == c].mean()), glif_hz=float(r_all_g[cls == c].mean())) for c in CLASSES}
for c in CLASSES:
    per_class[c]["diff_pct"] = 100 * (per_class[c]["hybrid_hz"] / per_class[c]["glif_hz"] - 1)

rh, rg = rates(hyb, T0, T1)[is_pc], rates(glif, T0, T1)[is_pc]
dist = {"hybrid": dist_stats(rh), "GLIF network": dist_stats(rg), "in vivo (Neuropixels L2/3 RS)": dist_stats(invivo)}
at_invivo_n = {"hybrid": at_n(rh, len(invivo)), "GLIF network": at_n(rg, len(invivo))}
act_h, act_v = rh[rh >= SILENT], invivo[invivo >= SILENT]
ks_shape = float(ks_2samp(np.log10(act_h / np.median(act_h)), np.log10(act_v / np.median(act_v))).statistic)

summary = dict(condition=CONDITION, gain=cond["gain"], pc_to_in_gain=PC_TO_IN_GAIN, tstop_ms=TSTOP_MS,
               population=cond["population"], front_jit=FRONT_JIT, gpu=str(tf.config.list_physical_devices("GPU")),
               wall_min=run_info["wall_s"] / 60, timers_s=run_info["timers_s"],
               per_class_rates_whole_run=per_class, pc_distribution_grating=dist, at_invivo_sample_size=at_invivo_n,
               ks_shape_active_log_rates_vs_invivo=ks_shape,
               pc_rate_vs_glif=dict(pearson=float(pearsonr(rh, rg)[0]), spearman=float(spearmanr(rh, rg)[0])))

if cond["population"]:
    pop = json.load(open(cond["population"]))
    assign = np.load(os.path.join(os.path.dirname(cond["population"]), pop["assignment"]))
    names = [m["name"] for m in pop["models"]]
    per_model = {names[k]: dict(n_pc=int((assign == k).sum()), mean_hz=float(rh[assign == k].mean()),
                                frac_silent=float((rh[assign == k] < SILENT).mean()),
                                share_of_spikes=float(rh[assign == k].sum() / rh.sum()))
                 for k in range(len(names)) if (assign == k).any()}
    summary["per_model_grating"] = per_model

import pandas as pd
print("Rates per class, 0-%d ms (Hz)" % T_END)
display(pd.DataFrame(per_class).T.round(2))
print(f"PC rate distribution, {T0:.0f}-{T1:.0f} ms")
display(pd.DataFrame(dist).T[["n", "mean", "median", "frac_silent", "p90", "p99", "max", "top10_share", "top1_share",
                               "sd_log10_active", "skew_log10_active"]].round(3))
print("At the in vivo sample size (595 cells): [2.5%, median, 97.5%]")
display(pd.DataFrame(at_invivo_n).T)
print("Per-PC rate vs GLIF network:", {k: round(v, 3) for k, v in summary["pc_rate_vs_glif"].items()},
      "| KS distance of active log-rate shape vs in vivo:", round(ks_shape, 3))
print("Run time (min): total %.1f, PCs %.1f, NEST %.1f, routing %.1f" % (
    summary["wall_min"], *(run_info["timers_s"][k] / 60 for k in ("pc", "nest", "route"))))
if cond["population"]:
    display(pd.DataFrame(per_model).T.sort_values("mean_hz", ascending=False).round(3))

## Step 9. Figures

1. **Overview**: raster of 400 random PCs and 100 cells of each interneuron class, and the population rate of
   each class over time (10 ms bins), hybrid (solid) vs GLIF network (thin black).
2. **Rates per class**, hybrid vs GLIF network.
3. **PC rate distribution** during the grating: histogram of log rate (silent cells counted at the left), and
   the cumulative share of spikes against the fraction of cells, most active first (the top-10% share is where
   each curve crosses x = 0.1), for the hybrid run, the GLIF network and in vivo.
4. **Each PC's rate** in the hybrid run vs in the GLIF network.
5. **Run time**: where the wall time went.
6. For heterogeneous conditions: **mean rate per model**.

In [ ]:
FIG = os.path.join(OUT, "figures"); os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
g = np.random.default_rng(1)

# 1. overview
fig, (a0, a1) = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw=dict(height_ratios=[2.2, 1]))
row = 0
for c, k in (("L2/3 E", 400), ("L2/3 Pvalb", 100), ("L2/3 Sst", 100), ("L2/3 Htr3a", 100)):
    pick = g.choice(np.flatnonzero(cls == c), k, replace=False)
    lut = np.full(N_L23, -1); lut[pick] = np.arange(k) + row
    m = lut[hyb[0]] >= 0
    a0.scatter(hyb[1][m], lut[hyb[0][m]], s=0.6, color=COLORS[c], lw=0)
    a0.text(T_END * 1.005, row + k / 2, c.replace("L2/3 ", ""), color=COLORS[c], va="center")
    row += k
a0.set(ylabel="cell", ylim=(0, row), xlim=(0, T_END * 1.06), title=f"{CONDITION}: raster (hybrid run)")
bins = np.arange(0, T_END + 10, 10)
for c in CLASSES:
    nc = (cls == c).sum()
    for sp, kw in ((hyb, dict(color=COLORS[c], lw=1.2)), (glif, dict(color="k", lw=0.5, alpha=0.6))):
        h = np.histogram(sp[1][cls[sp[0]] == c], bins)[0] / nc / 0.01
        a1.plot(bins[:-1] + 5, h, **kw)
if T_END > 500:                        # grating onset
    for ax in (a0, a1):
        ax.axvline(500, color="0.6", lw=0.8, ls="--")
a1.set(xlabel="time (ms)", ylabel="rate (Hz)", title="population rate per class (colour: hybrid, black: GLIF network)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "1_overview.png"), dpi=150); plt.show()

# 2. rates per class
fig, ax = plt.subplots(figsize=(5, 3.2))
x = np.arange(4)
ax.bar(x - 0.2, [per_class[c]["glif_hz"] for c in CLASSES], 0.4, color="0.25", label="GLIF network")
ax.bar(x + 0.2, [per_class[c]["hybrid_hz"] for c in CLASSES], 0.4, color=[COLORS[c] for c in CLASSES], label="hybrid")
for xi, c in zip(x, CLASSES):
    ax.text(xi + 0.2, per_class[c]["hybrid_hz"], f"{per_class[c]['diff_pct']:+.0f}%", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x, [c.replace("L2/3 ", "") for c in CLASSES]); ax.set(ylabel=f"mean rate 0-{T_END:.0f} ms (Hz)", title="Rates per class")
ax.legend(frameon=False); fig.tight_layout(); fig.savefig(os.path.join(FIG, "2_rates_per_class.png"), dpi=150); plt.show()

# 3. PC rate distribution
fig, (a0, a1) = plt.subplots(1, 2, figsize=(10, 3.6))
edges = np.r_[-1.0, np.linspace(np.log10(SILENT), 1.6, 25)]
sets = (("hybrid", rh, COLORS["L2/3 E"]), ("GLIF network", rg, "0.25"), ("in vivo", invivo, "#7d3c98"))
for nm, r, col in sets:
    lr = np.where(r >= SILENT, np.log10(np.maximum(r, 1e-9)), -0.7)
    a0.hist(lr, edges, density=False, weights=np.full(len(r), 1 / len(r)), histtype="step", lw=1.5, color=col, label=nm)
    a = np.sort(r)[::-1]
    a1.plot(np.arange(1, len(a) + 1) / len(a), np.cumsum(a) / a.sum(), color=col, lw=1.5,
            label=f"{nm}: top 10% fire {dist_stats(r)['top10_share']:.0%}")
a0.set_xticks([-0.7, 0, 0.5, 1, 1.5], ["silent", "1", "3.2", "10", "32"])
a0.set(xlabel=f"rate {T0:.0f}-{T1:.0f} ms (Hz)", ylabel="fraction of cells", title="PC rate distribution")
a0.legend(frameon=False)
a1.axvline(0.1, color="0.6", lw=0.8, ls="--")
a1.set(xlabel="fraction of cells (most active first)", ylabel="share of spikes", title="Who fires the spikes")
a1.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "3_pc_rate_distribution.png"), dpi=150); plt.show()

# 4. per-PC rates vs GLIF
fig, ax = plt.subplots(figsize=(4.2, 4))
j = g.uniform(-0.15, 0.15, (2, len(rh))) / ((T1 - T0) / 1000)
ax.scatter(rg + 0.1 + j[0], rh + 0.1 + j[1], s=1, alpha=0.15, color=COLORS["L2/3 E"], lw=0)
lim = [0.08, max(rh.max(), rg.max()) * 1.3]
ax.plot(lim, lim, color="0.5", lw=0.8)
ax.set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="GLIF network rate + 0.1 (Hz)", ylabel="hybrid rate + 0.1 (Hz)",
       title=f"Each PC (r = {summary['pc_rate_vs_glif']['pearson']:.2f}, ρ = {summary['pc_rate_vs_glif']['spearman']:.2f})")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "4_pc_rate_vs_glif.png"), dpi=150); plt.show()

# 5. run time
fig, ax = plt.subplots(figsize=(5, 2.2))
tm = run_info["timers_s"]
parts = [("PC layer (GPU)", tm["pc"]), ("NEST interneurons", tm["nest"]), ("spike routing", tm["route"]),
         ("setup and saving", run_info["wall_s"] - sum(tm.values()))]
left = 0
for (nm, v), col in zip(parts, ["#c0392b", "#2471a3", "#229954", "0.6"]):
    ax.barh(0, v / 60, left=left, color=col, label=f"{nm}: {v / 60:.1f} min"); left += v / 60
ax.set(yticks=[], xlabel="minutes", title=f"Run time for {TSTOP_MS} ms: {run_info['wall_s'] / 60:.1f} min")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.45), ncol=2)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "5_run_time.png"), dpi=150, bbox_inches="tight"); plt.show()

# 6. per model
if cond["population"]:
    pm = pd.DataFrame(per_model).T.sort_values("mean_hz", ascending=False)
    fig, ax = plt.subplots(figsize=(max(5, 0.22 * len(pm)), 3))
    ax.bar(np.arange(len(pm)), pm["mean_hz"], color=COLORS["L2/3 E"])
    ax.set_xticks(np.arange(len(pm)), pm.index, rotation=90, fontsize=7)
    ax.set(ylabel=f"mean rate {T0:.0f}-{T1:.0f} ms (Hz)", title="PCs of each fitted model")
    fig.tight_layout(); fig.savefig(os.path.join(FIG, "6_rate_per_model.png"), dpi=150); plt.show()

## Step 10. Save the statistics and figures to Drive
Everything ends up in `MyDrive/hybrid_l23_colab/runs/<condition>_<tstop>ms_<date>/`: `spikes.h5`, `run.json`
(gain, rates, timers), `log.txt`, `summary.json` (all statistics above) and `figures/`.

In [ ]:
with open(os.path.join(OUT, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
shutil.copytree(OUT, OUT_DRIVE, dirs_exist_ok=True)
print("saved to", OUT_DRIVE)
print(json.dumps({k: summary[k] for k in ("condition", "gain", "wall_min", "pc_rate_vs_glif")}, indent=2))